In [7]:
import os
from pathlib import Path

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER


# ============================================================
# 1. 用户设置区
# ============================================================

file_path = Path(
    "/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/data/2025JanFeb.nc"
)

# 总输出目录
output_base_dir = Path(
    "/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection/daily"
)

# 手动选择起始时间
# 注意：必须是数据中存在的时刻，例如 "2025-01-12T00:00:00"
start_time = "2025-02-03T00:00:00"

# 需要绘制的预报时效
forecast_hours = [0, 12, 24]

# 地图范围：东亚区域
map_extent = [70, 150, 20, 60]

# 绘图参数
pressure_level = 500.0
g = 9.80665
dpi = 300

# 是否允许自动选择最接近的时间
use_nearest_time = False


# ============================================================
# 2. 读取数据
# ============================================================

ds = xr.open_dataset(file_path)

print("=== 数据集基本信息 ===")
print(f"维度: {list(ds.dims)}")
print(f"坐标: {list(ds.coords)}")
print(f"变量: {list(ds.data_vars)}")

# 自动识别常见坐标名称
time_name = "valid_time" if "valid_time" in ds.coords else "time"
lat_name = "latitude" if "latitude" in ds.coords else "lat"
lon_name = "longitude" if "longitude" in ds.coords else "lon"

if "pressure_level" in ds.coords:
    level_name = "pressure_level"
elif "level" in ds.coords:
    level_name = "level"
else:
    raise KeyError("没有找到气压层坐标，请检查数据中的 pressure_level 或 level 名称。")

print(f"\n时间范围: {ds[time_name].values[0]} 至 {ds[time_name].values[-1]}")
print(f"时间数量: {len(ds[time_name])}")
print(f"气压层: {ds[level_name].values}")
print(f"纬度范围: {float(ds[lat_name].min()):.1f} 至 {float(ds[lat_name].max()):.1f}")
print(f"经度范围: {float(ds[lon_name].min()):.1f} 至 {float(ds[lon_name].max()):.1f}")


# ============================================================
# 3. 提取 500 hPa 位势高度
# ============================================================

if "z" not in ds.data_vars:
    raise KeyError("没有找到变量 z。请确认数据中的位势变量名称是否为 z。")

# ERA5 中 z 通常是 geopotential，单位为 m^2 s^-2，需要除以 g 转为 gpm
z_500 = ds["z"].sel({level_name: pressure_level}, method="nearest") / g

lat = ds[lat_name].values
lon = ds[lon_name].values

print(f"\n500 hPa 位势高度场形状: {z_500.shape}")


# ============================================================
# 4. 生成目标时间：起始时间、+12 h、+24 h
# ============================================================

start_np = np.datetime64(start_time, "ns")
target_times = [
    start_np + np.timedelta64(h, "h")
    for h in forecast_hours
]

available_times = ds[time_name].values.astype("datetime64[ns]")

selected_times = []

for t in target_times:
    if t in available_times:
        selected_times.append(t)
    else:
        if use_nearest_time:
            idx = np.argmin(np.abs(available_times - t))
            nearest_t = available_times[idx]
            selected_times.append(nearest_t)
            print(f"警告：目标时间 {t} 不存在，已使用最近时间 {nearest_t}")
        else:
            nearest_idx = np.argmin(np.abs(available_times - t))
            nearest_t = available_times[nearest_idx]
            raise ValueError(
                f"\n目标时间 {t} 不在数据中。\n"
                f"最接近的可用时间是 {nearest_t}。\n"
                f"如果你确认可以使用最近时间，请把 use_nearest_time 改为 True。"
            )

print("\n将绘制以下 3 个时次：")
for h, t in zip(forecast_hours, selected_times):
    print(f"  T+{h:02d} h: {t}")


# ============================================================
# 5. 固定色标范围
# ============================================================

z_min, z_max = 5000, 5900

fill_levels = np.arange(z_min, z_max, 20)
contour_levels = np.arange(z_min, z_max, 40)

print(f"\n统一色标范围: {z_min:.0f} 至 {z_max:.0f} gpm")


# ============================================================
# 6. 创建对应起始日期的输出子文件夹
# ============================================================

start_date_label = np.datetime_as_string(start_np, unit="D")
output_dir = output_base_dir / start_date_label
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\n图片将保存至：{output_dir}")


# ============================================================
# 7. 提前提取三幅图的数据
# ============================================================

z_data_list = []

for valid_time in selected_times:
    z_now = (
        z_500
        .sel({time_name: valid_time})
        .squeeze(drop=True)
        .transpose(lat_name, lon_name)
        .values
    )
    z_data_list.append(z_now)


# ============================================================
# 8. 单张图绘图函数
# ============================================================

def add_map_features(ax):
    ax.set_extent(map_extent, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.9)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.5, linestyle=":")
    ax.add_feature(cfeature.LAKES.with_scale("50m"), facecolor="none", edgecolor="gray", linewidth=0.4)
    ax.add_feature(cfeature.RIVERS.with_scale("50m"), edgecolor="gray", linewidth=0.3, alpha=0.5)

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.4,
        color="gray",
        alpha=0.6,
        linestyle=":"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 9}
    gl.ylabel_style = {"size": 9}
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER


def plot_z500_single(z_now, valid_time, lead_hour, start_time_str, save_path):
    """
    绘制单张 500 hPa 位势高度图
    """
    proj = ccrs.PlateCarree()

    fig = plt.figure(figsize=(11, 8))
    ax = plt.axes(projection=proj)

    cf = ax.contourf(
        lon,
        lat,
        z_now,
        levels=fill_levels,
        cmap="RdBu_r",
        extend="both",
        transform=proj
    )

    cs = ax.contour(
        lon,
        lat,
        z_now,
        levels=contour_levels,
        colors="black",
        linewidths=1.0,
        transform=proj
    )

    ax.clabel(
        cs,
        inline=True,
        fontsize=8,
        fmt="%d"
    )

    add_map_features(ax)

    valid_time_str = np.datetime_as_string(valid_time, unit="h").replace("T", " ")
    start_time_display = start_time_str.replace("T", " ")

    ax.set_title(
        f"ERA5 500 hPa Geopotential Height\n"
        f"Start: {start_time_display} UTC    Valid: {valid_time_str} UTC    T+{lead_hour:02d} h",
        fontsize=14,
        fontweight="bold",
        pad=14
    )

    cbar = fig.colorbar(
        cf,
        ax=ax,
        orientation="horizontal",
        pad=0.07,
        shrink=0.82,
        aspect=35
    )
    cbar.set_label("Geopotential Height at 500 hPa (gpm)", fontsize=11)
    cbar.ax.tick_params(labelsize=9)

    plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 9. 三联图绘图函数（横向排布）
# ============================================================

def plot_z500_triptych(z_data_list, selected_times, forecast_hours, start_time_str, save_path):
    """
    绘制 1×3 横向排布的大图
    """
    proj = ccrs.PlateCarree()

    fig, axes = plt.subplots(
        1, 3,
        figsize=(24, 7.5),
        subplot_kw={"projection": proj}
    )

    # 先手动给子图和 colorbar 预留空间
    fig.subplots_adjust(
        left=0.04,
        right=0.98,
        top=0.84,
        bottom=0.20,
        wspace=0.16
    )

    cf = None
    start_time_display = start_time_str.replace("T", " ")

    for ax, z_now, valid_time, lead_hour in zip(axes, z_data_list, selected_times, forecast_hours):

        cf = ax.contourf(
            lon,
            lat,
            z_now,
            levels=fill_levels,
            cmap="RdBu_r",
            extend="both",
            transform=proj
        )

        cs = ax.contour(
            lon,
            lat,
            z_now,
            levels=contour_levels,
            colors="black",
            linewidths=0.9,
            transform=proj
        )

        ax.clabel(
            cs,
            inline=True,
            fontsize=7,
            fmt="%d"
        )

        add_map_features(ax)

        valid_time_str = np.datetime_as_string(valid_time, unit="h").replace("T", " ")

        ax.set_title(
            f"T+{lead_hour:02d} h\n{valid_time_str} UTC",
            fontsize=12,
            fontweight="bold",
            pad=10
        )

    fig.suptitle(
        f"ERA5 500 hPa Geopotential Height\nStart: {start_time_display} UTC",
        fontsize=16,
        fontweight="bold",
        y=0.96
    )

    # 手动指定 colorbar 位置，避免被 tight layout 挤掉
    # [left, bottom, width, height]
    cbar_ax = fig.add_axes([0.22, 0.08, 0.56, 0.035])

    cbar = fig.colorbar(
        cf,
        cax=cbar_ax,
        orientation="horizontal"
    )

    cbar.set_label("Geopotential Height at 500 hPa (gpm)", fontsize=12)
    cbar.ax.tick_params(labelsize=10)

    # 这里不要再使用 bbox_inches="tight"
    plt.savefig(save_path, dpi=dpi)
    plt.close(fig)



# ============================================================
# 10. 先绘制三张单独图
# ============================================================

print("\n开始绘制三张单独图片...")

for z_now, lead_hour, valid_time in zip(z_data_list, forecast_hours, selected_times):

    start_label = np.datetime_as_string(start_np, unit="h").replace("-", "").replace("T", "_")
    valid_label = np.datetime_as_string(valid_time, unit="h").replace("-", "").replace("T", "_")

    filename = f"z500_start_{start_label}_T{lead_hour:02d}_valid_{valid_label}.png"
    save_path = output_dir / filename

    plot_z500_single(
        z_now=z_now,
        valid_time=valid_time,
        lead_hour=lead_hour,
        start_time_str=start_time,
        save_path=save_path
    )

    print(f"  已保存单图: {save_path}")


# ============================================================
# 11. 再绘制一个三联大图
# ============================================================

print("\n开始绘制三联大图...")

start_label = np.datetime_as_string(start_np, unit="h").replace("-", "").replace("T", "_")
triptych_path = output_dir / f"z500_triptych_start_{start_label}.png"

plot_z500_triptych(
    z_data_list=z_data_list,
    selected_times=selected_times,
    forecast_hours=forecast_hours,
    start_time_str=start_time,
    save_path=triptych_path
)

print(f"  已保存三联图: {triptych_path}")

print(f"\n完成！共生成 4 张图，保存位置：{output_dir}")

=== 数据集基本信息 ===
维度: ['valid_time', 'pressure_level', 'latitude', 'longitude']
坐标: ['number', 'valid_time', 'pressure_level', 'latitude', 'longitude', 'expver']
变量: ['z', 'u', 'v']

时间范围: 2025-01-01T00:00:00.000000000 至 2025-02-28T23:00:00.000000000
时间数量: 1416
气压层: [500.]
纬度范围: -90.0 至 90.0
经度范围: 0.0 至 359.8

500 hPa 位势高度场形状: (1416, 721, 1440)

将绘制以下 3 个时次：
  T+00 h: 2025-02-03T00:00:00.000000000
  T+12 h: 2025-02-03T12:00:00.000000000
  T+24 h: 2025-02-04T00:00:00.000000000

统一色标范围: 5000 至 5900 gpm

图片将保存至：/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection/daily/2025-02-03

开始绘制三张单独图片...
  已保存单图: /Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection/daily/2025-02-03/z500_start_20250203_00_T00_valid_20250203_00.png
  已保存单图: /Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection/daily/2025-02-03/z500_start_20250203_00_T12_valid_20250203_12.png
  已保存单图: /Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection/daily